# Proyecto Final Data Analyst — Análisis Integral de Datos de Clientes y Ventas

**Mayo 2026 · Bloque VI**

## Objetivo General

Realizar un análisis completo de un dataset de clientes y ventas, integrando técnicas de:
- Análisis exploratorio de datos (EDA)
- Análisis descriptivo y visualización
- Clasificación (predicción de abandono)
- Regresión (predicción de ventas)
- Clustering (segmentación de clientes)
- Series temporales (análisis de tendencias)

## Dataset
- **Archivo**: `../data/clientes_ventas_integral_2025.csv`
- **Registros**: 500 transacciones de clientes
- **Período**: Enero - Diciembre 2025
- **Variables**: 17 características (demográficas, comportamiento, ventas, satisfacción)

## 1. Configuración e Importación de Librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Librerías de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score
)
from sklearn.decomposition import PCA

## 2. Carga y Exploración Inicial

In [ ]:
df = pd.read_csv('../data/clientes_ventas_integral_2025.csv')
df['fecha'] = pd.to_datetime(df['fecha'])

print(f"Dimensiones: {df.shape}")
print(f"\nPrimeras filas:")
display(df.head())
print(f"\nTipos de datos:")
print(df.dtypes)

In [ ]:
print("\nEstadísticas descriptivas:")
df.describe()

In [ ]:
print("\nValores faltantes:")
print(df.isnull().sum())

print("\nDistribuciones categóricas:")
print(f"Segmentos: {df['segmento'].value_counts().to_dict()}")
print(f"Regiones: {df['region'].value_counts().to_dict()}")
print(f"Canales: {df['canal'].value_counts().to_dict()}")

## 3. Análisis Descriptivo y Visualización

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Ventas por segmento
df.groupby('segmento')['venta'].sum().plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Ventas totales por Segmento')
axes[0, 0].set_ylabel('Ventas ($)')

# Abandono por segmento
df.groupby('segmento')['abandono'].mean().plot(kind='bar', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Tasa de Abandono por Segmento')
axes[0, 1].set_ylabel('Tasa de Abandono')

# Ventas por región
df.groupby('region')['venta'].mean().plot(kind='bar', ax=axes[0, 2], color='lightgreen')
axes[0, 2].set_title('Venta promedio por Región')
axes[0, 2].set_ylabel('Venta promedio ($)')

# Distribución de ventas
axes[1, 0].hist(df['venta'], bins=30, color='purple', alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Distribución de Ventas')
axes[1, 0].set_xlabel('Venta ($)')

# Satisfacción por canal
df.boxplot(column='satisfaction', by='canal', ax=axes[1, 1])
axes[1, 1].set_title('Satisfacción por Canal')
axes[1, 1].set_ylabel('Satisfacción (1-5)')

# Productos más vendidos
df['producto'].value_counts().plot(kind='barh', ax=axes[1, 2], color='teal')
axes[1, 2].set_title('Cantidad de transacciones por Producto')
axes[1, 2].set_xlabel('Cantidad')

plt.tight_layout()
plt.show()

## 4. Análisis de Correlaciones

In [ ]:
# Seleccionar variables numéricas
df_numeric = df[['cantidad', 'precio_unitario', 'descuento', 'venta', 'abandono', 
                   'satisfaction', 'compras_previas', 'dias_desde_ultima', 
                   'ticket_promedio', 'reclamaciones', 'devolucion']]

# Matriz de correlación
correlacion = df_numeric.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlacion, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            cbar_kws={'label': 'Correlación'})
plt.title('Matriz de Correlaciones')
plt.tight_layout()
plt.show()

print("\nCorrelaciones con Abandono:")
print(correlacion['abandono'].sort_values(ascending=False))

## 5. Clasificación: Predicción de Abandono

In [ ]:
# Preparación de datos
df_model = df.copy()

# Codificación de variables categóricas
le_segmento = LabelEncoder()
le_region = LabelEncoder()
le_canal = LabelEncoder()
le_producto = LabelEncoder()

df_model['segmento_enc'] = le_segmento.fit_transform(df_model['segmento'])
df_model['region_enc'] = le_region.fit_transform(df_model['region'])
df_model['canal_enc'] = le_canal.fit_transform(df_model['canal'])
df_model['producto_enc'] = le_producto.fit_transform(df_model['producto'])

# Features y target
features_clf = ['cantidad', 'descuento', 'satisfaction', 'compras_previas', 
                 'dias_desde_ultima', 'ticket_promedio', 'reclamaciones', 'devolucion',
                 'segmento_enc', 'region_enc', 'canal_enc', 'producto_enc']

X = df_model[features_clf]
y = df_model['abandono']

# Separación train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Proporción de abandono en train: {y_train.mean():.2%}")
print(f"Proporción de abandono en test: {y_test.mean():.2%}")

In [ ]:
# Entrenamiento del modelo de clasificación
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# Predicciones
y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)[:, 1]

# Métricas
print("\n═══ CLASIFICACIÓN: Predicción de Abandono ═══")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.3f}")

print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['No Abandono', 'Abandono']))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Abandono', 'Abandono'],
            yticklabels=['No Abandono', 'Abandono'])
plt.title('Matriz de Confusión - Predicción de Abandono')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.show()

## 6. Regresión: Predicción de Ventas

In [ ]:
# Features y target para regresión
features_reg = ['cantidad', 'descuento', 'compras_previas', 'dias_desde_ultima', 
                 'satisfaction', 'reclamaciones', 'segmento_enc', 'canal_enc']

X_reg = df_model[features_reg]
y_reg = df_model['venta']

# Separación
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Entrenamiento
reg = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
reg.fit(X_train_reg, y_train_reg)

# Predicciones
y_pred_reg = reg.predict(X_test_reg)

# Métricas
mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

print("\n═══ REGRESIÓN: Predicción de Ventas ═══")
print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²:   {r2:.3f}")
print(f"\nVentas reales (test): media ${y_test_reg.mean():,.2f}, std ${y_test_reg.std():,.2f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.5, s=50)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 
         'r--', lw=2, label='Predicción perfecta')
plt.xlabel('Ventas Reales ($)')
plt.ylabel('Ventas Predichas ($)')
plt.title('Regresión: Ventas Reales vs Predichas')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. Clustering: Segmentación de Clientes

In [ ]:
# Preparación de datos para clustering
features_cluster = ['venta', 'satisfaction', 'compras_previas', 'dias_desde_ultima', 
                     'ticket_promedio', 'abandono', 'descuento']

X_cluster = df_model[features_cluster].copy()

# Escalado
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Encontrar el número óptimo de clusters (elbow method)
inercias = []
silhuetas = []
K_range = range(2, 11)

for k in K_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_temp = kmeans_temp.fit_predict(X_scaled)
    inercias.append(kmeans_temp.inertia_)
    silhuetas.append(silhouette_score(X_scaled, labels_temp))

# Visualizar el codo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_range, inercias, 'bo-', linewidth=2)
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método del Codo')
axes[0].grid(alpha=0.3)

axes[1].plot(K_range, silhuetas, 'ro-', linewidth=2)
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Coeficiente de Silhueta')
axes[1].set_title('Análisis de Silhueta')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Coeficientes de Silhueta por K:")
for k, sil in zip(K_range, silhuetas):
    print(f"  K={k}: {sil:.3f}")

In [ ]:
# Usar k=3 como óptimo
k_optimo = 3
kmeans = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df_model['cluster'] = clusters

print(f"\n═══ CLUSTERING: Segmentación en {k_optimo} Clusters ═══")
print(f"Distribución: {np.bincount(clusters).to_dict()}")
print(f"Coeficiente de Silhueta: {silhouette_score(X_scaled, clusters):.3f}")

In [ ]:
# Visualización de clusters con PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', 
                       s=100, alpha=0.6, edgecolors='black')
plt.scatter(pca.transform(scaler.transform(kmeans.cluster_centers_))[:, 0],
           pca.transform(scaler.transform(kmeans.cluster_centers_))[:, 1],
           marker='X', s=500, c='red', edgecolors='black', linewidths=2, label='Centroides')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title(f'Clustering K-Means (K={k_optimo}) - Visualización con PCA')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Caracterización de clusters
print("\nCaracterización de Clusters:")
for cluster_id in range(k_optimo):
    cluster_data = df_model[df_model['cluster'] == cluster_id]
    print(f"\n--- Cluster {cluster_id} (n={len(cluster_data)}) ---")
    print(f"  Venta promedio: ${cluster_data['venta'].mean():,.2f}")
    print(f"  Satisfaction: {cluster_data['satisfaction'].mean():.2f}/5")
    print(f"  Compras previas: {cluster_data['compras_previas'].mean():.1f}")
    print(f"  Tasa de abandono: {cluster_data['abandono'].mean():.2%}")
    print(f"  Descuento promedio: {cluster_data['descuento'].mean():.2%}")

## 8. Series Temporales: Análisis de Tendencias

In [ ]:
# Agregación temporal
df_temporal = df.groupby(df['fecha'].dt.to_period('W')).agg({
    'venta': 'sum',
    'cantidad': 'sum',
    'abandono': 'mean',
    'satisfaction': 'mean',
    'cliente_id': 'nunique'
}).rename(columns={'cliente_id': 'clientes_unicos'})

df_temporal.index = df_temporal.index.to_timestamp()

print("Evolución semanal:")
display(df_temporal.head(10))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Ventas totales
axes[0, 0].plot(df_temporal.index, df_temporal['venta'], marker='o', color='steelblue', linewidth=2)
axes[0, 0].set_title('Ventas Semanales')
axes[0, 0].set_ylabel('Venta Total ($)')
axes[0, 0].grid(alpha=0.3)

# Cantidad de transacciones
axes[0, 1].plot(df_temporal.index, df_temporal['cantidad'], marker='s', color='green', linewidth=2)
axes[0, 1].set_title('Cantidad de Transacciones Semanales')
axes[0, 1].set_ylabel('Cantidad')
axes[0, 1].grid(alpha=0.3)

# Tasa de abandono
axes[1, 0].plot(df_temporal.index, df_temporal['abandono']*100, marker='^', color='red', linewidth=2)
axes[1, 0].set_title('Tasa de Abandono Semanal')
axes[1, 0].set_ylabel('Abandono (%)')
axes[1, 0].grid(alpha=0.3)

# Satisfacción promedio
axes[1, 1].plot(df_temporal.index, df_temporal['satisfaction'], marker='d', color='purple', linewidth=2)
axes[1, 1].set_title('Satisfacción Promedio Semanal')
axes[1, 1].set_ylabel('Satisfaction (1-5)')
axes[1, 1].set_ylim([0, 5])
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Importancia de Features

In [ ]:
# Importancias del modelo de clasificación
importancias_clf = pd.DataFrame({
    'feature': features_clf,
    'importancia': clf.feature_importances_
}).sort_values('importancia', ascending=False)

# Importancias del modelo de regresión
importancias_reg = pd.DataFrame({
    'feature': features_reg,
    'importancia': reg.feature_importances_
}).sort_values('importancia', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(importancias_clf['feature'][:10], importancias_clf['importancia'][:10], color='steelblue')
axes[0].set_title('Top 10 Features - Clasificación (Abandono)')
axes[0].set_xlabel('Importancia')

axes[1].barh(importancias_reg['feature'][:10], importancias_reg['importancia'][:10], color='coral')
axes[1].set_title('Top 10 Features - Regresión (Ventas)')
axes[1].set_xlabel('Importancia')

plt.tight_layout()
plt.show()

## 10. Recomendaciones y Conclusiones

In [ ]:
print("""
═══════════════════════════════════════════════════════════════════
                   CONCLUSIONES Y RECOMENDACIONES
═══════════════════════════════════════════════════════════════════

1. ANÁLISIS DESCRIPTIVO
   • Total de transacciones: 500
   • Venta promedio: ${:,.2f}
   • Tasa de abandono: {:.1%}
   • Satisfacción media: {:.2f}/5

2. PREDICCIÓN DE ABANDONO (Clasificación)
   • Accuracy: {:.1%}
   • Recall (clientes en riesgo identificados): {:.1%}
   • Acción: Implementar estrategias de retención para clientes identificados

3. PREDICCIÓN DE VENTAS (Regresión)
   • R² Score: {:.3f}
   • Error promedio: ${:,.2f}
   • Acción: Usar para proyecciones y asignación de recursos

4. SEGMENTACIÓN DE CLIENTES (Clustering)
   • Clusters identificados: {}
   • Acción: Aplicar estrategias diferenciadas por cluster

5. ANÁLISIS TEMPORAL
   • Rango: {} a {}
   • Patrón: Monitorear tendencias semanales

6. RECOMENDACIONES ESTRATÉGICAS
   ✓ Enfoque en satisfacción de cliente (correlación fuerte con abandono)
   ✓ Optimizar descuentos por segmento
   ✓ Incrementar engagement con clientes de bajo valor
   ✓ Monitoreo continuo de KPIs
   ✓ Implementar modelo de predicción en producción

═══════════════════════════════════════════════════════════════════
""".format(
    df['venta'].mean(),
    df['abandono'].mean(),
    df['satisfaction'].mean(),
    accuracy_score(y_test, y_pred),
    recall_score(y_test, y_pred),
    r2_score(y_test_reg, y_pred_reg),
    mean_absolute_error(y_test_reg, y_pred_reg),
    k_optimo,
    df['fecha'].min().date(),
    df['fecha'].max().date()
))